# 🧩 MATA Graph System — Advanced Pipelines

Go beyond sequential chains — build **parallel**, **conditional**, and
**multi-model** vision workflows with the MATA graph system.

This notebook covers:
- **Parallel execution** — run independent tasks concurrently
- **Conditional branching** — `If` nodes with predicates
- **Multi-task fusion** — detect + classify + depth in one pass
- **Custom predicates** — write your own routing logic
- **VLM integration** — vision-language models in a graph
- **Presets** — pre-built pipelines for common patterns

**Prerequisites**: `pip install datamata[notebook]`

> **⚠️ Vision-language models (VLMs) use a lot of computational resources**
>
> May require computational resources beyond typical detection/segmentation models.
> so be mindful of your hardware capabilities when running VLMs.
>
> Dev notes: Tested on RTX 3070 with 8GB VRAM, so be mindful of your hardware capabilities when running VLMs.


In [1]:
# Uncomment to install if needed:
# !pip install datamata[notebook]
# !pip install timm

In [2]:
import mata
from pathlib import Path
print(f"MATA version: {mata.__version__}")

IMAGE_1   = "../../examples/images/000000039769.jpg"
IMAGE_2   = "../../examples/images/000000015338.jpg"

for label, p in [("image 1", IMAGE_1), ("image 2", IMAGE_2)]:
    status = "FOUND" if Path(p).exists() else "NOT FOUND"
    print(f"  {label:12s}: {status}")

MATA version: 1.9.4
  image 1     : FOUND
  image 2     : FOUND


## 1️⃣ Parallel Detection + Classification + Depth

Run **three independent tasks in parallel** for 1.5–3× speedup.
Use `Graph.parallel()` + `ParallelScheduler` to execute concurrently.

In [3]:
from mata.core.graph import Graph, ParallelScheduler
from mata.nodes import Classify, Detect, EstimateDepth, Filter, Fuse

# Load providers
detector = mata.load("detect", "facebook/detr-resnet-50")
classifier = mata.load("classify", "openai/clip-vit-base-patch32")
depth_model = mata.load("depth", "depth-anything/Depth-Anything-V2-Small-hf")

# Build a parallel graph
graph = (
    Graph("full_scene")
    .parallel([
        Detect(using="detector", out="dets"),
        Classify(using="classifier", text_prompts=["indoor", "outdoor"], out="cls"),
        EstimateDepth(using="depth", out="depth"),
    ])
    .then(Filter(src="dets", score_gt=0.3, out="filtered"))
    .then(Fuse(dets="filtered", classification="cls", depth="depth", out="scene"))
)

result = mata.infer(
    image=IMAGE_1,
    graph=graph,
    providers={
        "detector": detector,
        "classifier": classifier,
        "depth": depth_model,
    },
    scheduler=ParallelScheduler(),
)

print(f"Scene type: {result.scene.classification.top1.label_name}")
print(f"Objects:    {len(result.scene.dets.instances)}")
print(f"Depth map:  {result.scene.depth.shape}")

[INFO] Loading detect model from huggingface: facebook/detr-resnet-50
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceDetectAdapter with device=cuda, threshold=0.3


d:\Documents\OneDrive\Code\mtp\datamata_io\mata\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Loading HuggingFace model: facebook/detr-resnet-50
[INFO] Detected architecture: detr
[INFO] Model loaded successfully on cuda
[INFO] Loading classify model from huggingface: openai/clip-vit-base-patch32
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceClassifyAdapter with device=cuda, threshold=0.0
[INFO] Loading HuggingFace classification model: openai/clip-vit-base-patch32
[INFO] Detected architecture: clip
[INFO] Routing to CLIP zero-shot classification adapter
[INFO] Using specified device: cuda
[INFO] Initialized HuggingFaceCLIPAdapter with device=cuda, threshold=0.0
[INFO] Using template: a photo of a {}
[INFO] Loading CLIP model: openai/clip-vit-base-patch32
[INFO] Successfully loaded CLIP model on device: cuda
[INFO] Loading depth model from huggingface: depth-anything/Depth-Anything-V2-Small-hf
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceDepthAdapter with device=cuda, threshold=0.0
[IN

In [ ]:
# Visualise — detection overlay + depth colormap
mata.show(result.scene.dets.to_vision_result(), image=IMAGE_1)
mata.show(result.scene.depth.to_depth_result())

## 2️⃣ Detection → Parallel Segmentation + Depth

Mix sequential and parallel stages: detect first, then segment and
estimate depth **in parallel** on the detected regions.

In [4]:
from mata.core.graph import Graph, ParallelScheduler
from mata.nodes import Detect, Filter, PromptBoxes, EstimateDepth, Fuse

segmenter = mata.load("segment", "facebook/sam-vit-base")

graph = (
    Graph("detect_then_parallel")
    .then(Detect(using="detector", out="dets"))
    .then(Filter(src="dets", score_gt=0.5, out="filtered"))
    .parallel([
        PromptBoxes(using="segmenter", dets_src="filtered", out="masks"),
        EstimateDepth(using="depth", out="depth"),
    ])
    .then(Fuse(dets="filtered", masks="masks", depth="depth", out="complete"))
)

result = mata.infer(
    IMAGE_2,
    graph=graph,
    providers={
        "detector": detector,
        "segmenter": segmenter,
        "depth": depth_model,
    },
    scheduler=ParallelScheduler(),
)

print(f"Segmented + depth for {len(result.complete.dets.instances)} objects")

[INFO] Loading segment model from huggingface: facebook/sam-vit-base
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceSAMAdapter with device=cuda, threshold=0.0
[INFO] Loading SAM model: facebook/sam-vit-base
[INFO] Loaded SAM model on cuda (zero-shot mode, RLE=True)
[INFO] ParallelScheduler initialized with 4 workers
[INFO] Starting parallel execution of graph 'detect_then_parallel' with 4 workers
[INFO] Running SAM on 640x424 image (boxes=6)
[INFO] Generated 18 masks (threshold=0.00)
[INFO] Graph 'detect_then_parallel' completed in 582.08ms (5 nodes, 3 stages)
Segmented + depth for 6 objects


In [ ]:
# Visualise — segmentation masks + depth colormap on IMAGE_2
mata.show(result.complete.masks.to_vision_result(), image=IMAGE_2)
mata.show(result.complete.depth.to_depth_result())

## 3️⃣ Conditional Execution — Segment Only If Objects Found

Skip expensive segmentation when detection returns nothing.
Use `If` with a built-in `CountAbove` predicate.

In [5]:
from mata.core.graph import Graph, If, CountAbove, Pass
from mata.nodes import Detect, Filter, PromptBoxes, Fuse

graph = (
    Graph("conditional_segment")
    .then(Detect(using="detector", out="dets"))
    .then(Filter(src="dets", score_gt=0.5, out="filtered"))
    .then(If(
        predicate=CountAbove("filtered", 0),
        then_branch=PromptBoxes(using="segmenter", dets_src="filtered", out="masks"),
        else_branch=Pass(),
    ))
    .then(Fuse(dets="filtered", out="final"))
)

result = mata.infer(
    IMAGE_1,
    graph=graph,
    providers={"detector": detector, "segmenter": segmenter},
)

print(f"Objects found: {len(result.final.dets.instances)}")

[INFO] Starting synchronous execution of graph 'conditional_segment'
[INFO] Graph 'conditional_segment' completed in 119.89ms (4 nodes)
Objects found: 5


In [ ]:
# Visualise — filtered detections (segmentation skipped when nothing found)
mata.show(result.final.dets.to_vision_result(), image=IMAGE_1)

## 4️⃣ Quality-Based Routing

Route execution based on detection confidence:
- **High confidence** (score > 0.8) → keep top 5 only
- **Low confidence** → fall back to a looser filter

In [6]:
from mata.core.graph import Graph, If, ScoreAbove
from mata.nodes import Detect, TopK, Filter, Fuse

graph = (
    Graph("quality_routing")
    .then(Detect(using="detector", out="dets"))
    .then(If(
        predicate=ScoreAbove("dets", 0.8),
        then_branch=TopK(k=5, src="dets", out="final_dets"),
        else_branch=Filter(src="dets", score_gt=0.3, out="final_dets"),
    ))
    .then(Fuse(dets="final_dets", out="final"))
)

result = mata.infer(
    IMAGE_1,
    graph=graph,
    providers={"detector": detector},
)

for inst in result.final.dets.instances:
    print(f"  {inst.label_name}: {inst.score:.2f}")

[INFO] Starting synchronous execution of graph 'quality_routing'
[INFO] Graph 'quality_routing' completed in 76.30ms (3 nodes)
  cat: 1.00
  cat: 1.00
  remote: 1.00
  remote: 1.00
  couch: 1.00


In [ ]:
# Visualise — quality-routed detection results
mata.show(result.final.dets.to_vision_result(), image=IMAGE_1)

## 5️⃣ Label-Conditional Segmentation

Only segment when a **specific class** is detected — e.g., segment cats only.

In [19]:
from mata.core.graph import Graph, If, HasLabel, Pass
from mata.nodes import Detect, PromptBoxes, Fuse

graph = (
    Graph("cat_segmenter")
    .then(Detect(using="detector", out="dets"))
    .then(If(
        predicate=HasLabel("dets", "cat"),
        #then_branch=Fuse(dets="dets", out="final"),
        then_branch=PromptBoxes(using="segmenter", dets_src="dets", out="masks"),
        else_branch=Pass(),
    ))
    .then(Fuse(dets="dets", masks="masks", out="final"))
)

result = mata.infer(
    IMAGE_1,
    graph=graph,
    providers={"detector": detector, "segmenter": segmenter},
)

result.final.masks
print(f"Result: {len(result.final.dets.instances)} instance(s)")
print(f"Result: {len(result.final.masks.instances)} instance(s)")

[INFO] Starting synchronous execution of graph 'cat_segmenter'
[INFO] Running SAM on 640x480 image (boxes=5)
[INFO] Generated 15 masks (threshold=0.00)
[INFO] Graph 'cat_segmenter' completed in 5991.83ms (3 nodes)
Result: 5 instance(s)
Result: 5 instance(s)


In [ ]:
# Visualise — cat segmentation masks (when cats detected)
mata.show(result.final.masks.to_vision_result(), image=IMAGE_1)

## 6️⃣ Custom Predicate Function

Write your own routing logic — e.g., segment only when a large object is detected.

In [20]:
from mata.core.graph import Graph, If, Pass
from mata.nodes import Detect, PromptBoxes, Fuse


def has_large_objects(ctx):
    """Check if any detection covers >20% of image area."""
    dets = ctx.retrieve("dets")
    image = ctx.retrieve("input.image")
    img_area = image.width * image.height

    for inst in dets.instances:
        if inst.bbox is not None:
            x1, y1, x2, y2 = inst.bbox
            box_area = (x2 - x1) * (y2 - y1)
            if box_area / img_area > 0.2:
                return True
    return False


graph = (
    Graph("large_object_handler")
    .then(Detect(using="detector", out="dets"))
    .then(If(
        predicate=has_large_objects,
        then_branch=PromptBoxes(using="segmenter", dets_src="dets", out="masks"),
        else_branch=Pass(),
    ))
    .then(Fuse(dets="dets", masks="masks", out="final"))
)

result = mata.infer(
    IMAGE_1,
    graph=graph,
    providers={"detector": detector, "segmenter": segmenter},
)

print(f"Processed: {len(result.final.dets.instances)} objects")

[INFO] Starting synchronous execution of graph 'large_object_handler'
[INFO] Running SAM on 640x480 image (boxes=5)
[INFO] Generated 15 masks (threshold=0.00)
[INFO] Graph 'large_object_handler' completed in 5942.50ms (3 nodes)
Processed: 5 objects


In [ ]:
# Visualise — large-object detections (with masks when predicate triggers)
mata.show(result.final.dets.to_vision_result(), image=IMAGE_1)

## 7️⃣ VLM in a Graph — Scene Description

Use a **Vision-Language Model** node for semantic scene understanding.

In [9]:
# Load a vision-language model for potential future use in the graph, it will used a lot of VRAM (depending on the model, of course) so be careful. Make sure to run this cell only once if you are in a notebook environment.
# it took about 12s on RTX 3070 8GB to load Qwen3-VL-2B-Instruct
vlm = mata.load("vlm", "Qwen/Qwen3-VL-2B-Instruct")

[INFO] Loading vlm model from huggingface: Qwen/Qwen3-VL-2B-Instruct
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceVLMAdapter with device=cuda, threshold=0.3
[INFO] Loading VLM model: Qwen/Qwen3-VL-2B-Instruct
[INFO] Loaded VLM model: Qwen/Qwen3-VL-2B-Instruct on cuda


In [21]:
from mata.nodes import VLMQuery, Fuse

result = mata.infer(
    image=IMAGE_1,
    graph=[
        VLMQuery(
            using="vlm",
            prompt="Describe this image in detail.",
            output_mode="describe",
            out="description",
        ),
        Fuse(description="description", out="final"),
    ],
    providers={"vlm": vlm},
)

print(result.final.description.meta.get("text", ""))


[INFO] Starting synchronous execution of graph 'untitled_graph'
[INFO] Parsed 5 entities from VLM output (mode=describe)
[INFO] Graph 'untitled_graph' completed in 15380.56ms (2 nodes)
```json
{
  "description": "A close-up shot of two cats sleeping on a bright pink couch. The cat on the left is a tabby with black and brown stripes, lying on its back with its paws extended. The cat on the right is a tabby with a lighter brown and black pattern, lying on its side. Both cats appear relaxed and are surrounded by two remote controls, one white and one gray, placed on the couch. The couch is covered in a soft, pink fabric.",
  "objects": [
    {
      "label": "cat",
      "confidence": 0.95
    },
    {
      "label": "cat",
      "confidence": 0.95
    },
    {
      "label": "remote control",
      "confidence": 0.95
    },
    {
      "label": "remote control",
      "confidence": 0.95
    },
    {
      "label": "couch",
      "confidence": 0.95
    }
  ],
  "scene": "a pink couch with

In [ ]:
# Visualise — show source image alongside VLM description
from IPython.display import Image as IPImage
IPImage(IMAGE_1, width=400)

Detections(instances=[], instance_ids=[], entities=[Entity(label='cat', score=0.95, attributes={}), Entity(label='cat', score=0.95, attributes={}), Entity(label='remote control', score=0.95, attributes={}), Entity(label='remote control', score=0.95, attributes={}), Entity(label='couch', score=0.98, attributes={})], entity_ids=['ent_0000', 'ent_0001', 'ent_0002', 'ent_0003', 'ent_0004'], meta={'model_id': 'Qwen/Qwen3-VL-2B-Instruct', 'device': 'cuda', 'backend': 'huggingface', 'max_new_tokens': 512, 'tokens_generated': 239, 'image_path': 'd:\\Documents\\OneDrive\\Code\\mtp\\datamata_io\\mata\\examples\\notebooks\\..\\..\\examples\\images\\000000039769.jpg', 'image_paths': ['d:\\Documents\\OneDrive\\Code\\mtp\\datamata_io\\mata\\examples\\notebooks\\..\\..\\examples\\images\\000000039769.jpg'], 'image_count': 1, 'image_width': 640, 'image_height': 480, 'output_mode': 'describe', 'text': '```json\n{\n  "description": "Two cats are sleeping on a bright pink couch. The cat on the left is a 

## 8️⃣ VLM + Detection — Grounded Scene Analysis

Combine a VLM for semantic understanding with GroundingDINO for spatial
localization, then fuse both into a single result.

In [22]:
from mata.core.graph import Graph, ParallelScheduler
from mata.nodes import VLMQuery, Detect, Filter, Fuse

grounding_dino = mata.load("detect", "IDEA-Research/grounding-dino-tiny")

graph = (
    Graph("vlm_grounded_scene")
    .parallel([
        VLMQuery(
            using="vlm",
            prompt="What objects are in this image?",
            output_mode="describe",
            out="scene_desc",
        ),
        Detect(using="grounding_dino", text_prompts="cat . remote . couch", out="dets"),
    ])
    .then(Filter(src="dets", score_gt=0.3, out="filtered"))
    .then(Fuse(dets="filtered", description="scene_desc", out="final"))
)

result = mata.infer(
    IMAGE_1,
    graph=graph,
    providers={"vlm": vlm, "grounding_dino": grounding_dino},
    scheduler=ParallelScheduler(),
)

print("VLM says:", result.final.description.meta.get("text", ""))
print(f"Detected: {len(result.final.dets.instances)} objects spatially")


[INFO] Loading detect model from huggingface: IDEA-Research/grounding-dino-tiny
[INFO] Auto-selected CUDA device (NVIDIA GeForce RTX 3070)
[INFO] Initialized HuggingFaceZeroShotDetectAdapter with device=cuda, threshold=0.3
[INFO] Detected zero-shot architecture: grounding_dino
[INFO] Loading GroundingDINO model: IDEA-Research/grounding-dino-tiny
[INFO] Successfully loaded grounding_dino on cuda
[INFO] ParallelScheduler initialized with 4 workers
[INFO] Starting parallel execution of graph 'vlm_grounded_scene' with 4 workers


d:\Documents\OneDrive\Code\mtp\datamata_io\mata\.venv\Lib\site-packages\transformers\models\grounding_dino\processing_grounding_dino.py:91: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)


[INFO] Parsed 4 entities from VLM output (mode=describe)
[INFO] Graph 'vlm_grounded_scene' completed in 16822.25ms (4 nodes, 3 stages)
VLM says: ```json
{
  "description": "Two cats are lying on a bright pink couch, sleeping. One cat is on the left, with a black and gray striped pattern, and the other is on the right, with a brown and black striped pattern. Both cats are resting with their eyes closed. There are two remote controls on the couch, one white and one with a blue button, next to the cats.",
  "objects": [
    {
      "label": "cat",
      "confidence": 0.94
    },
    {
      "label": "cat",
      "confidence": 0.92
    },
    {
      "label": "remote control",
      "confidence": 0.96
    },
    {
      "label": "remote control",
      "confidence": 0.94
    }
  ],
  "scene": "couch"
}
```
Detected: 4 objects spatially


In [ ]:
# Visualise — GroundingDINO detection overlay with grounded objects
mata.show(result.final.dets.to_vision_result(), image=IMAGE_1)

Detections(instances=[], instance_ids=[], entities=[Entity(label='cat', score=0.95, attributes={}), Entity(label='cat', score=0.93, attributes={}), Entity(label='remote control', score=0.96, attributes={}), Entity(label='remote control', score=0.94, attributes={})], entity_ids=['ent_0000', 'ent_0001', 'ent_0002', 'ent_0003'], meta={'model_id': 'Qwen/Qwen3-VL-2B-Instruct', 'device': 'cuda', 'backend': 'huggingface', 'max_new_tokens': 512, 'tokens_generated': 126, 'image_path': 'd:\\Documents\\OneDrive\\Code\\mtp\\datamata_io\\mata\\examples\\notebooks\\..\\..\\examples\\images\\000000039769.jpg', 'image_paths': ['d:\\Documents\\OneDrive\\Code\\mtp\\datamata_io\\mata\\examples\\notebooks\\..\\..\\examples\\images\\000000039769.jpg'], 'image_count': 1, 'image_width': 640, 'image_height': 480, 'output_mode': 'describe', 'text': '```json\n{\n  "description": "Two cats sleeping on a pink couch with remote controls beside them.",\n  "objects": [\n    {\n      "label": "cat",\n      "confidenc

## 9️⃣ Using Presets — One-Liner Pipelines

MATA ships pre-built graph presets for common patterns.

In [23]:
from mata.presets import grounding_dino_sam

# GroundingDINO + SAM: text-prompted detection → segmentation
result = mata.infer(
    image=IMAGE_1,
    graph=grounding_dino_sam(
        detection_threshold=0.3,
        text_prompts="cat . remote . couch",
    ),
    providers={
        "detector": grounding_dino,
        "segmenter": segmenter,
    },
)

print(f"Preset result: {len(result.final.dets.instances)} segmented objects")

[INFO] Starting synchronous execution of graph 'grounding_dino_sam'
[INFO] Running SAM on 640x480 image (boxes=4)
[INFO] Generated 12 masks (threshold=0.00)
[INFO] Graph 'grounding_dino_sam' completed in 9259.15ms (5 nodes)
Preset result: 4 segmented objects


In [ ]:
# Visualise — GroundingDINO + SAM segmentation masks
mata.show(result.final.masks.to_vision_result(), image=IMAGE_1)

Masks(instances=[Instance(score=0.9971538782119751, label=0, bbox=(347.0, 19.0, 639.0, 277.0), mask={'size': [480, 640], 'counts': '[iR5=c>0O2Hg0_O2^Oe0^EXNd7h1\\HXNc7j1[HWNW7`2gFnMa0Eg8_2fFnMa0Db8i2dFoMb0YOj8h2dFoMb0YOi8j2cFoMb0XOh8^4kFeKT9]4jFdKS9V500O100O1M210O1K5O1O100O1O001N110O1O1O1O101N1M4N1O2O0O1O100O100O1O1O1O100O1O1O1O100O1O1O1O101N1O2O00000O100O1O100O1O10000O100O10000O100O0100O10000O100O0010O0100O1000O010O0100000000000000O1O1O1O10OM4O1E:010OM4O1O1O100O1O1M2010O0100O10001O001O01O001O100O010O1O1O1O001O1O1OgMhGkMW8T2kGkMU8U2kGlMU8P2oGoMQ8P2QHPNn7o1THQNk7n1WHQNi7o1WHRNi7h1^HWNa7h1aHXN_7e1dH[N[7d1gH[NY7e1gH\\NX7b1kH`NR7_1PIaNP7^1QIbNo6\\1SIcNm6]1SIdNm6Z1WIdNi6Z1YIfNg6X1\\IhNb6W1`IhNa6W1_IjN`6T1cInNZ6Q1gIPOX6P1iIPOV6o0lIPOT6P1lIQOT6l0oIVOn5i0SJXOm5d0YJ[Oe5d0]J[Oc5e0]J\\Oc5b0_JUNjM9f7a1bJTNjM:e7^1eJWNgM:d7^1gJVNgM<a7^1hJVNgM<b7W1oJ[NaM=_7W1RK[N`M>^7o0YKcNZM`0Z7l0^KdNXM?[7l0]KeNXM`0Z7j0`KfNVM`0Y7i0bKgNVM`0W7g0fKhNTMa0U7f0hKiNTM`0T7g0iKhNSMb0T7c0kKkNTM`0P7d0mKlNTM?P7d0lKmNWM<m6f0mKnN

## 🔟 Full Scene Analysis Preset

Detect + classify + depth in one call using the `full_scene_analysis` preset.

In [25]:
from mata.presets import full_scene_analysis
from mata.core.graph import ParallelScheduler

graph = full_scene_analysis(
    detection_threshold=0.3,
    classification_labels=["indoor", "outdoor", "urban", "nature"],
)

result = mata.infer(
    image=IMAGE_1,
    graph=graph,
    providers={
        "detector": detector,
        "classifier": classifier,
        "depth": depth_model,
    },
    scheduler=ParallelScheduler(),
)

result

[INFO] ParallelScheduler initialized with 4 workers
[INFO] Starting parallel execution of graph 'full_scene' with 4 workers
[INFO] Graph 'full_scene' completed in 1115.92ms (5 nodes, 3 stages)


MultiResult(channels=[depth, class, dets, filtered, scene], provenance=7 keys, metrics=9 keys)

In [26]:
print(f"Scene: {result.scene.classifications.top1.label_name}")
print(f"Objects: {len(result.scene.dets.instances)}")

Scene: indoor
Objects: 5


In [ ]:
# Visualise — detection overlay + depth colormap from full scene analysis
mata.show(result.scene.dets.to_vision_result(), image=IMAGE_1)
mata.show(result.scene.depth.to_depth_result())

## Summary

| Pattern | Key API | Use case |
|---------|---------|----------|
| **Parallel** | `Graph.parallel()` + `ParallelScheduler` | Independent tasks (detect + classify + depth) |
| **Conditional** | `If(predicate, then, else)` | Skip expensive steps, quality routing |
| **Predicates** | `CountAbove`, `ScoreAbove`, `HasLabel`, custom `fn(ctx)` | Route execution dynamically |
| **VLM nodes** | `VLMQuery`, `VLMDetect` | Semantic understanding in a graph |
| **Presets** | `grounding_dino_sam`, `full_scene_analysis` | Common workflows as one-liners |

**Docs**: See `docs/GRAPH_COOKBOOK.md` and `docs/GRAPH_API_REFERENCE.md` for the full API.